In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "catalog_ecobici_dev")
catalogo = dbutils.widgets.get("catalogo")

In [0]:
G_DIM_FECHA    = f"{catalogo}.golden.dim_fecha"
G_DIM_ESTACION = f"{catalogo}.golden.dim_estacion"
G_DIM_CLIMA    = f"{catalogo}.golden.dim_clima"
G_FACT_CLIMA   = f"{catalogo}.golden.fact_clima_hora"
G_FACT_VIAJES  = f"{catalogo}.golden.fact_viajes"

In [0]:
A_DEMANDA    = f"{catalogo}.golden.agg_demanda_hora_dia"
A_ESTACIONES = f"{catalogo}.golden.agg_estaciones"
A_RUTAS      = f"{catalogo}.golden.agg_rutas_frecuentes"
A_CLIMA      = f"{catalogo}.golden.agg_uso_por_clima"
A_DIARIO     = f"{catalogo}.golden.agg_resumen_diario"

In [0]:
def guardar(df, tabla):
    (
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(tabla)
    )
    print(f"{tabla}: {spark.table(tabla).count():,} filas")

In [0]:
fact         = spark.table(G_FACT_VIAJES)
dim_fecha    = spark.table(G_DIM_FECHA)
dim_estacion = spark.table(G_DIM_ESTACION)
dim_clima    = spark.table(G_DIM_CLIMA)
clima_hora   = spark.table(G_FACT_CLIMA)
n_viajes     = fact.count()
print(f"Viajes en fact_viajes: {n_viajes:,}")

Viajes en fact_viajes: 6,065,074


1. Demanda por día de la semana y hora

In [0]:
demanda = (
    fact
    .join(dim_fecha.select("fecha_key", "dia_semana_num", "nombre_dia"), "fecha_key")
    .groupBy("dia_semana_num", "nombre_dia", "hora_retiro")
    .agg(
        F.count("*").alias("viajes"),
        F.countDistinct("fecha_key").alias("dias_con_viajes"),
        F.round(F.avg("duracion_min"), 2).alias("duracion_media_min"),
    )
    .withColumn("viajes_promedio_por_dia", F.round(F.col("viajes") / F.col("dias_con_viajes"), 1))
    .orderBy("dia_semana_num", "hora_retiro")
)
guardar(demanda, A_DEMANDA)

catalog_ecobici_dev.golden.agg_demanda_hora_dia: 141 filas


2. Balance por estación (retiros contra arribos)
Balance neto = arribos - retiros
Positivo: la estación acumula bicicletas
Negativo: la estación esta vacía

In [0]:
retiros = fact.groupBy(F.col("estacion_retiro_key").alias("estacion_key")).agg(F.count("*").alias("retiros"))
arribos = fact.groupBy(F.col("estacion_arribo_key").alias("estacion_key")).agg(F.count("*").alias("arribos"))

estaciones = (
    dim_estacion.select(
        "estacion_key", "id_ubicacion", "nombre_estacion", "colonia", "alcaldia", "latitud", "longitud"
    )
    .join(retiros, "estacion_key", "left")
    .join(arribos, "estacion_key", "left")
    .na.fill(0, ["retiros", "arribos"])
    .withColumn("balance_neto", F.col("arribos") - F.col("retiros"))
    .withColumn("viajes_totales", F.col("arribos") + F.col("retiros"))
    .orderBy("estacion_key")
)
guardar(estaciones, A_ESTACIONES)

catalog_ecobici_dev.golden.agg_estaciones: 689 filas


3. Rutas mas frecuentes

In [0]:
origen = dim_estacion.select(
    F.col("estacion_key").alias("estacion_retiro_key"),
    F.col("nombre_estacion").alias("origen"),
    F.col("colonia").alias("colonia_origen"),
)
destino = dim_estacion.select(
    F.col("estacion_key").alias("estacion_arribo_key"),
    F.col("nombre_estacion").alias("destino"),
    F.col("colonia").alias("colonia_destino"),
)

rutas = (
    fact.groupBy("estacion_retiro_key", "estacion_arribo_key")
    .agg(F.count("*").alias("viajes"), F.round(F.avg("duracion_min"), 2).alias("duracion_media_min"))
    .orderBy(F.desc("viajes"))
    .limit(200)
    .join(origen, "estacion_retiro_key", "left")
    .join(destino, "estacion_arribo_key", "left")
    .withColumn("ranking", F.row_number().over(Window.orderBy(F.desc("viajes"), "estacion_retiro_key", "estacion_arribo_key")))
    .select(
        "ranking", "estacion_retiro_key", "origen", "colonia_origen",
        "estacion_arribo_key", "destino", "colonia_destino", "viajes", "duracion_media_min",
    )
    .orderBy("ranking")
)
guardar(rutas, A_RUTAS)

catalog_ecobici_dev.golden.agg_rutas_frecuentes: 200 filas


4. Uso según el clima, viajes por hora de cada condición

In [0]:
limites = fact.agg(F.min("ts_retiro").alias("primero"), F.max("ts_retiro").alias("ultimo")).first()

horas_por_clima = (
    clima_hora
    .filter(
        F.col("fecha_hora").between(
            F.date_trunc("hour", F.lit(limites["primero"])),
            F.date_trunc("hour", F.lit(limites["ultimo"])),
        )
    )
    .groupBy("clima_key")
    .agg(F.count("*").alias("horas_en_periodo"))
)

uso_clima = (
    fact.groupBy("clima_key")
    .agg(F.count("*").alias("viajes"), F.round(F.avg("duracion_min"), 2).alias("duracion_media_min"))
    .join(horas_por_clima, "clima_key", "left")
    .join(dim_clima, "clima_key", "left")
    .withColumn("viajes_por_hora", F.round(F.col("viajes") / F.col("horas_en_periodo"), 1))
    .select(
        "clima_key", "descripcion_clima", "llueve", "rango_temperatura",
        "viajes", "horas_en_periodo", "viajes_por_hora", "duracion_media_min",
    )
    .orderBy("clima_key")
)
guardar(uso_clima, A_CLIMA)

catalog_ecobici_dev.golden.agg_uso_por_clima: 29 filas


5. Resumen diario

In [0]:
clima_dia = (
    clima_hora.join(dim_clima.select("clima_key", "llueve"), "clima_key", "left")
    .groupBy("fecha_key")
    .agg(
        F.round(F.avg("temperatura_c"), 1).alias("temperatura_media_c"),
        F.round(F.sum("precipitacion_mm"), 1).alias("precipitacion_total_mm"),
        F.sum(F.col("llueve").cast("int")).alias("horas_con_lluvia"),
    )
)

diario = (
    fact.groupBy("fecha_key")
    .agg(F.count("*").alias("viajes"), F.round(F.avg("duracion_min"), 2).alias("duracion_media_min"))
    .join(dim_fecha.select("fecha_key", "fecha", "nombre_dia", "es_fin_de_semana"), "fecha_key", "left")
    .join(clima_dia, "fecha_key", "left")
    .select(
        "fecha_key", "fecha", "nombre_dia", "es_fin_de_semana", "viajes", "duracion_media_min",
        "temperatura_media_c", "precipitacion_total_mm", "horas_con_lluvia",
    )
    .orderBy("fecha_key")
)
guardar(diario, A_DIARIO)

catalog_ecobici_dev.golden.agg_resumen_diario: 124 filas


6. Verificaciones

In [0]:
def total(tabla, columna):
    return spark.table(tabla).agg(F.sum(columna)).first()[0]


assert total(A_DEMANDA, "viajes") == n_viajes, "agg_demanda_hora_dia no suma todos los viajes"
assert total(A_CLIMA, "viajes") == n_viajes, "agg_uso_por_clima no suma todos los viajes"
assert total(A_DIARIO, "viajes") == n_viajes, "agg_resumen_diario no suma todos los viajes"
assert total(A_ESTACIONES, "retiros") == n_viajes, "agg_estaciones: los retiros no suman todos los viajes"
assert total(A_ESTACIONES, "arribos") == n_viajes, "agg_estaciones: los arribos no suman todos los viajes"
assert total(A_ESTACIONES, "balance_neto") == 0, "El balance neto de todas las estaciones debe sumar cero"
print("Todos los totales cuadran con fact_viajes:", f"{n_viajes:,}")

Todos los totales cuadran con fact_viajes: 6,065,074


In [0]:
display(spark.table(A_CLIMA))

clima_key,descripcion_clima,llueve,rango_temperatura,viajes,horas_en_periodo,viajes_por_hora,duracion_media_min
-1,Sin dato,false,Sin dato,783,null,null,38.76
1,Despejado,false,Cálido (24°C o más),38647,11,3513.4,15.92
2,Despejado,false,Fresco (12 a 18°C),94451,126,749.6,14.38
4,Despejado,false,Templado (18 a 24°C),153564,55,2792.1,16.06
5,Llovizna intensa,true,Fresco (12 a 18°C),72461,42,1725.3,15.66
6,Llovizna intensa,true,Templado (18 a 24°C),31891,9,3543.4,15.59
7,Llovizna ligera,true,Cálido (24°C o más),255552,74,3453.4,15.55
8,Llovizna ligera,true,Fresco (12 a 18°C),667409,449,1486.4,14.56
9,Llovizna ligera,true,Frío (menos de 12°C),190,1,190.0,13.63
10,Llovizna ligera,true,Templado (18 a 24°C),1071944,337,3180.8,15.84


In [0]:
display(spark.table(A_ESTACIONES).orderBy(F.desc("viajes_totales")).limit(20))

estacion_key,id_ubicacion,nombre_estacion,colonia,alcaldia,latitud,longitud,retiros,arribos,balance_neto,viajes_totales
271,271,Jesus Garcia y Carlos J. Meneses,Buenavista,Cuauhtemoc,19.443684,-99.152465,47089,70407,23318,117496
548,548,Doctor Mariano Azuela y Jose Antonio Alzate,Santa Maria La Ribera,Cuauhtemoc,19.44701,-99.154318,34121,36783,2662,70904
27,27,Av. Paseo De La Reforma y Havre,Juarez,Cuauhtemoc,19.42916,-99.162703,33624,33935,311,67559
64,64,Sonora y Amsterdam,Hipodromo,Cuauhtemoc,19.412982,-99.166936,30579,33126,2547,63705
38,38,Cozumel y Puebla,Roma Norte,Cuauhtemoc,19.4208433,-99.1713729,30985,30613,-372,61598
14,14,Av. Paseo De La Reforma y Rio De La Plata,Cuauhtemoc,Cuauhtemoc,19.42454,-99.173247,26417,31885,5468,58302
208,208,Hesiodo y Lamartine,Polanco,Miguel Hidalgo,19.434182,-99.189835,29855,26701,-3154,56556
555,555,Av. Ribera De San Cosme y Av. Insurgentes Centro,Santa Maria La Ribera,Cuauhtemoc,19.44021597,-99.15616778,28575,27625,-950,56200
18,18,Av. Paseo De La Reforma y Rio Rhin,Cuauhtemoc,Cuauhtemoc,19.428802,-99.164381,25770,27949,2179,53719
1,1,Rio Sena y Rio Balsas,Cuauhtemoc,Cuauhtemoc,19.4335895,-99.1678193,26545,27091,546,53636
